# 4주차 실습 — 헬스케어 데이터관리 · 실습본 (TODO 채우기)
**Kaggle FitBit Fitness Tracker Data 원자료**(`dailyActivity_merged.csv`, `sleepDay_merged.csv`)로 데이터 품질을 점검하고, **규칙 기반 알림**과 **AI 이상탐지(Isolation Forest)** 결과를 비교합니다.

> ⚠️ 교육용 분석입니다. 이상탐지 결과는 의료 진단이 아닙니다.

| 단계 | 내용 |
|---|---|
| 0 | 데이터 불러오기 (A: Kaggle 직접 / B: Drive 제공본 / C: GitHub URL) |
| 1~3 | 구조 확인 → 결측·중복·범위 점검 → 미착용 의심일 탐지 |
| 4~5 | 날짜형 변환 → Id+날짜로 병합 |
| 6~8 | 규칙 기반 알림 → Isolation Forest → 결과 비교 |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest

pd.set_option('display.max_columns', 30)

## 0. 데이터 불러오기
- **방법 A·B** (Kaggle 직접 다운로드 / Drive 제공본): 왼쪽 📁 파일 패널에 두 CSV를 업로드한 뒤 `USE_LOCAL_FILE = True`로 바꿉니다.
- **방법 C** (기본값): 업로드 없이 수업 GitHub 저장소에서 바로 읽습니다.

In [ ]:
USE_LOCAL_FILE = False   # 직접 업로드했다면 True

if USE_LOCAL_FILE:
    daily = pd.read_csv('dailyActivity_merged.csv')
    sleep = pd.read_csv('sleepDay_merged.csv')
else:
    BASE = 'https://raw.githubusercontent.com/itlects/2026-2_ai-data/main/Week04/data/'
    daily = pd.read_csv(BASE + 'dailyActivity_merged.csv')
    sleep = pd.read_csv(BASE + 'sleepDay_merged.csv')

print('daily shape =', daily.shape)   # 기대: (940, 15)
print('sleep shape =', sleep.shape)   # 기대: (413, 5)

## 1. 구조 확인

In [ ]:
print(daily.columns.tolist())
print(sleep.columns.tolist())
print('활동 사용자 수 =', daily['Id'].nunique(), '/ 수면 사용자 수 =', sleep['Id'].nunique())
display(daily.head(3))
display(sleep.head(3))

## 2. 품질 점검 ① 결측치와 중복

In [ ]:
print('[결측치 합계] daily =', daily.isna().sum().sum(), '/ sleep =', sleep.isna().sum().sum())
print('[완전 중복 행] daily =', daily.duplicated().sum(), '/ sleep =', sleep.duplicated().sum())
display(sleep[sleep.duplicated(keep=False)])

sleep_clean = sleep.drop_duplicates().copy()
print('sleep before =', len(sleep), '→ after =', len(sleep_clean))   # 413 → 410

## 3. 품질 점검 ② 범위·극단값과 ‘숨은 결측’
결측치가 0개여도 **걸음수 0 + 비활동 1,440분(=24시간)** 인 날은 기기를 차지 않은 날일 가능성이 높습니다.

In [ ]:
display(daily[['TotalSteps','VeryActiveMinutes','SedentaryMinutes','Calories']].describe().T)

non_wear = (daily['TotalSteps'] == 0) & (daily['SedentaryMinutes'] == 1440)
print('걸음수 0인 날        =', (daily['TotalSteps'] == 0).sum())
print('비활동 1440분인 날   =', (daily['SedentaryMinutes'] == 1440).sum())
print('미착용 의심(둘 다)   =', non_wear.sum())
print('칼로리 0인 날        =', (daily['Calories'] == 0).sum())

per_user = daily.groupby('Id').size()
print('31일 미만 기록 사용자 =', (per_user < 31).sum(), '명 / 최소 기록일 =', per_user.min())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(daily.loc[~non_wear, 'TotalSteps'], daily.loc[~non_wear, 'SedentaryMinutes'], s=10, alpha=0.4, label='normal record')
ax.scatter(daily.loc[non_wear, 'TotalSteps'], daily.loc[non_wear, 'SedentaryMinutes'], s=30, color='tab:red', label='non-wear suspected')
ax.axhline(1440, ls=':', color='tab:red')
ax.set_xlabel('TotalSteps'); ax.set_ylabel('SedentaryMinutes'); ax.legend()
plt.show()

## 4. 날짜형 변환
`SleepDay`는 `4/12/2016 12:00:00 AM` 형식이므로 **format을 명시**해야 경고 없이 정확히 변환됩니다.

In [ ]:
daily['ActivityDate'] = pd.to_datetime(daily['ActivityDate'], format='%m/%d/%Y')
sleep_clean['SleepDay'] = pd.to_datetime(sleep_clean['SleepDay'], format='%m/%d/%Y %I:%M:%S %p')

print(daily['ActivityDate'].min().date(), '~', daily['ActivityDate'].max().date())
print(daily['ActivityDate'].dtype, sleep_clean['SleepDay'].dtype)

## 5. 병합 (Id + 날짜)

In [ ]:
merged = pd.merge(
    daily, sleep_clean,
    left_on=['Id', 'ActivityDate'],
    right_on=['Id', 'SleepDay'],
    how='inner'
)
print('merged shape =', merged.shape)            # 기대: (410, 19)
print('병합 후 사용자 수 =', merged['Id'].nunique())  # 24

merged['SleepHours'] = merged['TotalMinutesAsleep'] / 60
analysis_df = merged[['Id','ActivityDate','TotalSteps','Calories',
                      'SedentaryMinutes','VeryActiveMinutes','SleepHours']].copy()
display(analysis_df.describe().T)

## 6. 규칙 기반 알림 (교육용 예시 기준)

In [ ]:
# TODO ⑥: rule_alert 함수를 작성하세요.
#  - TotalSteps < 1000 → 'low_steps'
#  - SleepHours < 4 → 'short_sleep',  SleepHours > 12 → 'long_sleep'
#  - 해당 없음 → 'normal'
# analysis_df['rule_alert'] = analysis_df.apply(rule_alert, axis=1)


## 7. AI 이상탐지 — Isolation Forest

In [ ]:
# TODO ⑦: features 5개로 IsolationForest(contamination=0.05, random_state=42)를 학습하고
#          analysis_df['ai_anomaly'] ('anomaly'/'normal')와 analysis_df['ai_score']를 만드세요.
features = ['TotalSteps', 'Calories', 'SedentaryMinutes', 'VeryActiveMinutes', 'SleepHours']


## 8. 규칙과 AI 결과 비교

In [ ]:
# TODO ⑧: rule_flag 컬럼을 만들고 pd.crosstab으로 규칙 vs AI 교차표를 만드세요.
#          '규칙 X / AI O', '규칙 O / AI X' 행을 각각 출력해 보세요.


In [ ]:
# (선택) 규칙·AI 결과를 산점도로 시각화해 보세요. 완성본 노트북 참고

## 9. 심화 — contamination 변화

In [ ]:
# TODO ⑨: contamination을 0.02~0.20으로 바꿔 가며 AI 이상 건수를 비교하세요.


## 10. 해석 질문
1. 결측치가 0개인데도 데이터 품질 문제가 있다고 말할 수 있는 근거는?
2. inner merge로 530일이 빠졌습니다. 이것이 분석 결과에 어떤 편향을 줄 수 있을까요?
3. 규칙 O / AI X 30건은 대부분 어떤 규칙 때문인가요? 이 날들은 정말 ‘이상’인가요?
4. 규칙 X / AI O 9건은 어떤 공통점이 있나요? (힌트: 걸음수·칼로리·고강도 활동)
5. 실제 헬스케어 서비스라면 알림을 보내기 전에 어떤 검증 단계가 필요할까요?